In [ ]:
# ============================================================================
# SPP constraint-family exposure build — fully inline (no prod-pipeline edits)
# Writes only to temp.* / *_SPP_test tables. Reads prod/dayzer/MySQL read-only.
# ============================================================================
import sys, warnings, re
warnings.filterwarnings("ignore")
sys.path.append("/var/www/python/Qingcheng/nighthawk/")

import pandas as pd, numpy as np
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
from datetime import datetime, timedelta
import pytz
from nighthawk.util import bigquery_functions
from nighthawk.util.sql_functions import download_df_from_sql_db
from nighthawk.data.network.constraint import Constraint
from nighthawk.data.network.constraint_family import ConstraintFamily

OPEX = "SPP"
SUFFIX = "_SPP_test"          # all SPP component tables get this suffix
DATASET = "constraint_family_exposure"
ICE_LABEL = "INDIANAHUB"      # SPP uses INDIANAHUB ice price as proxy (per decision)

bid_dt = (datetime.now(pytz.timezone("America/Chicago")) + timedelta(days=2)).strftime("%Y-%m-%d")
# bid_dt = "2026-06-09"
start_dt = (pd.to_datetime(bid_dt) - pd.Timedelta("1105 day")).strftime("%Y-%m-%d")  # ~3y lookback
end_dt   = bid_dt
mvalue_threshold = 100

print("bid_dt     :", bid_dt)
print("lookback   :", start_dt, "->", end_dt)
print("suffix     :", SUFFIX)


bid_dt     : 2026-06-17
lookback   : 2023-06-08 -> 2026-06-17
suffix     : _SPP_test


# Check for the dayzer flow ratio prediction vs next day actual DART value change. 

In [32]:
# --- Step A: dayzer ConstraintId -> internal oops_constraint_num mapping (SPP) ---
# SPP dayzer names embed the readable monitored line after a colon, e.g.
#   "8765B_LN@HAYMAKR4<13386>:LN HAYMAKR4 - CIMARRON"
# We parse that segment and normalize to match internal monitored_clean.
# Contingency names don't share a convention, so we match MONITORED-ONLY
# (covers ~92% of dayzer SPP CIDs). Approximate — flagged.

dz = bigquery_functions.download_df_from_bq(f"""
    SELECT DISTINCT CID AS ConstraintId, ISOConstraint AS monitored_name, ISOContingency AS contingency_name
    FROM `movetocloud-999.dayzer.ISOName`
    WHERE marketName='{OPEX}' AND CID > 0
""")
internal_da = Constraint(market=OPEX).get_constraint_details(da_or_rt="DA")[
    ["oops_constraint_num", "monitored_clean", "contingency_clean"]].rename(
    columns={"monitored_clean": "monitored_name", "contingency_clean": "contingency_name"})
internal_rt = Constraint(market=OPEX).get_constraint_details(da_or_rt="RT")[
    ["oops_constraint_num", "monitored_clean", "contingency_clean"]].rename(
    columns={"monitored_clean": "monitored_name", "contingency_clean": "contingency_name"})
internal_dart = pd.concat([internal_da, internal_rt])


def _spp_mon_key(s):
    s = str(s)
    if ":" in s:                       
        s = s.split(":", 1)[1]
    return s.lower().replace(" ", "").replace("-", "").replace("_", "")

dz["monitored_key"] = dz["monitored_name"].map(_spp_mon_key)
internal_dart["monitored_key"] = internal_dart["monitored_name"].map(_spp_mon_key)
# internal_rep = internal_dart.drop_duplicates(subset="monitored_key")   # one internal con per monitored key
internal_rep = internal_dart

dayzer_oops_mapping = pd.merge(
    internal_rep[["oops_constraint_num", "monitored_name", "contingency_name", "monitored_key"]],
    dz[["ConstraintId", "monitored_key"]], on="monitored_key", how="inner"
)

dayzer_oops_mapping.drop_duplicates(inplace=True)
dayzer_oops_mapping_bq = bigquery_functions.upload_to_bq_from_dataframe(
    dayzer_oops_mapping, "temp", f"{OPEX}_dayzer_oops_mapping{SUFFIX}", temp=True)

print("dayzer SPP CIDs total      :", dz["ConstraintId"].nunique())
print("mapped CIDs                :", dayzer_oops_mapping["ConstraintId"].nunique(),
      f"({100*dayzer_oops_mapping['ConstraintId'].nunique()/dz['ConstraintId'].nunique():.0f}%)")
print("mapping table              :", dayzer_oops_mapping_bq)
display(dayzer_oops_mapping.head())
print('Not mapped monitored_key at our constraint table are', len(internal_rep[~internal_rep['oops_constraint_num'].isin(dayzer_oops_mapping.oops_constraint_num)].monitored_key.unique())/len(internal_rep.monitored_key.unique()))

print('most of CID 92% already mapped, but 71% of the monitored key not mapped by dayzer, coverage is an issue, will need \
      to see if those monitored key actually happen very small then disregard it if not dayzer is a problem')

dayzer SPP CIDs total      : 2624
mapped CIDs                : 2414 (92%)
mapping table              : temp.SPP_dayzer_oops_mapping_SPP_test


,oops_constraint_num,monitored_name,contingency_name,monitored_key,ConstraintId
0,151242,ln-texas_co,base,lntexasco,1000070
1,683,ln134pentp-mcclai,lnmcclai-saraa138okge,ln134pentpmcclai,3020455
2,4378,ln134pentp-mcclai,lnmcclai-plvala138okge,ln134pentpmcclai,3020455
3,273357,ln134pentp-mcclai,okge:mcclaisara:138:1:7,ln134pentpmcclai,3020455
4,484992,ln134pentp-mcclai,okge:mcclais_lakes:138:1:8,ln134pentpmcclai,3020455


Not mapped monitored_key at our constraint table are 0.7134943496565478
most of CID 92% already mapped, but 71% of the monitored key not mapped by dayzer, coverage is an issue, will need       to see if those monitored key actually happen very small then disregard it if not dayzer is a problem


NT mapping has 932 mapped compared with 1294 mapped by monitored name. 596 important ones mapped compared with 681. 

In [33]:
# select those monitored names that has either da or rt mvalue smaller than -1000 
# 3-year window ending at bid_dt
start_3y = (pd.Timestamp(bid_dt) - pd.Timedelta(days=1095)).strftime("%Y-%m-%d")

# per-constraint summed mvalue over 3y (granularity=None -> overall sum per constraint)
mv_da = Constraint(oops_constraint_num_df=None, market=OPEX).get_mvalues(
    start_dt=start_3y, end_dt=bid_dt, type="DA", granularity=None)
mv_rt = Constraint(oops_constraint_num_df=None, market=OPEX).get_mvalues(
    start_dt=start_3y, end_dt=bid_dt, type="RT", granularity=None)

# map oops_constraint_num -> monitored_name, then sum per monitored
omap = internal_dart[["oops_constraint_num", "monitored_name"]].drop_duplicates("oops_constraint_num")
da_sum = (mv_da.merge(omap, on="oops_constraint_num", how="left")
          .groupby("monitored_name")["mvalue"].sum().rename("DA_mvalue"))
rt_sum = (mv_rt.merge(omap, on="oops_constraint_num", how="left")
          .groupby("monitored_name")["mvalue"].sum().rename("RT_mvalue"))

dart = pd.concat([da_sum, rt_sum], axis=1).fillna(0).reset_index()

# keep monitored with DA or RT 3y-sum below -1000
dart_select = dart[(dart["DA_mvalue"] < -1000) | (dart["RT_mvalue"] < -1000)].reset_index(drop=True)
dart_select = dart_select.sort_values(["DA_mvalue", "RT_mvalue"]).reset_index(drop=True)
print("monitored kept:", len(dart_select.monitored_name.unique()))


monitored kept: 1174


In [34]:
# whole-market mapping (ConstraintId <-> oops_constraint_num_da), contingency-aware
mapping_da = Constraint(oops_constraint_num_df=None, market=OPEX).get_dayzer_constraint_mapping_da()
mapping_rt = Constraint(oops_constraint_num_df=None, market=OPEX).get_dayzer_constraint_mapping_rt()
mapping = pd.concat([mapping_da,mapping_rt])
mapped_monitored = internal_dart[internal_dart['oops_constraint_num'].isin(mapping.oops_constraint_num_da)].monitored_name.unique()
print('mapped_monitored_name total are', len(mapped_monitored))
print('mapped in the dart -1000 are',dart_select[dart_select["monitored_name"].isin(mapped_monitored)].monitored_name.nunique())
unmapped_ones = set(mapped_monitored).difference(set(dart_select.monitored_name))
dart_unmapped = dart_select[~dart_select["monitored_name"].isin(mapped_monitored)].reset_index(drop=True)
dart_mapped = dart_select[dart_select["monitored_name"].isin(mapped_monitored)].reset_index(drop=True)

print('unmapped_monitored in dart smaller than -1000 are ', len(dart_unmapped))
print('mapped by joining by monitored name not constraint name and contingency name', len(dart_select[dart_select['monitored_name'].isin(dayzer_oops_mapping.monitored_name)].monitored_name.unique()))


mapped_monitored_name total are 939
mapped in the dart -1000 are 601
unmapped_monitored in dart smaller than -1000 are  573
mapped by joining by monitored name not constraint name and contingency name 681


In [35]:
dart_unmapped.sort_values(['RT_mvalue'])[:30]

,monitored_name,DA_mvalue,RT_mvalue
442,lnmagic_cy-souris,0.0000,-269433.7898
1,multi-elementconstraint.spsnmties.spsnmties,-171699.4671,-264069.0431
0,lngord-maiz,-198633.6054,-261181.5918
443,lnwin115-wprisn,0.0000,-158753.9933
15,xfmrnormhll-normhll,-39590.1219,-122270.4066
23,multi-elementconstraint.sppspsties.sppspsties,-35437.0692,-107353.9554
3,multi-elementconstraint.tmp623_29733.tmp623_29733,-100612.6993,-105328.5134
444,lnskst-idalia,0.0000,-101346.2898
445,lnkun07-tiog,0.0000,-93085.2360
11,lnsterling-sub974,-46483.2265,-73595.1039


In [36]:
print('mapped total RT select over the total RT select, ', dart_mapped.RT_mvalue.sum()/dart_select.RT_mvalue.sum())
print('mapped total DA select over the total RT select, ', dart_mapped.DA_mvalue.sum()/dart_select.DA_mvalue.sum())

mapped total RT select over the total RT select,  0.90247360325808
mapped total DA select over the total RT select,  0.8788794248738296


In [37]:
print('if not using the NT, the additional mapping would be', set(dart_select[dart_select['monitored_name'].isin(dayzer_oops_mapping.monitored_name)].monitored_name.unique()).difference(set(dart_select[dart_select["monitored_name"].isin(mapped_monitored)].monitored_name.unique())))

if not using the NT, the additional mapping would be {'lnteague-s_jal', 'lnwatford-wilistn2', 'lnn_loup-ord1', 'lnmoore_co-rbfaria', 'lnhoyt-norl', 'lnbyrd_sub-cooper_r', 'lnrckfalls-rdrunnr4', 'lnftrandl-bonestl', 'lnminot-loganwa', 'lnfrankln-hamp_tp', 'lnhoyt-54mr', 'lnbanntap4-mayo_rd', 'lnstan_tp-beulh7th', 'lnmoore_co-etter', 'lnsw_134tp-westmor4', 'lnfarb-ksc10', 'lnemcp-cjohn', 'lntblanca-hereford', 'lnft_cal-ft_cal', 'lndovers1-doversub', 'xfmrodessa1-odessa1', 'lnndprarwf-nelsontp', 'lncrofton-blmfld', 'xfmrnwhende-nwhende', 'lngord-maiz', 'xfmrblaisdel-blaisdel', 'lnfrankln5-litc', 'lnwatertn-wtr15ave', 'lngavins-spiritm', 'lnltlmiss-bakerwp', 'lnwafb_es-sed_mps', 'lnozrk_b-omaha', 'lnseminole-pittsb9', 'lnreeds_ed-brannw5', 'lnhebron4-mandan', 'lncanby-granitf', 'lnfairviwp-wilistn', 'lnrdrunner-battlaxe', 'lnogalala-brule', 'lnglendct-dawsonc', 'lntexas_co-', 'lnknob-marshal3', 'lncldoniaw-egf_ind', 'lndaycntyw-daycotap', 'lnnorton-c_marshl', 'lnbottnose-bottno', 'lnsummit

In [38]:
# oops_constraint_num -> constraint_family_num  (same method all markets use)
oops_family = Constraint(
    pd.DataFrame({"oops_constraint_num": mapping["oops_constraint_num_da"].unique()}),
    market=OPEX
).get_constraint_family_num()
# cols: oops_constraint_num, constraint_family_num

# attach family onto the dayzer mapping (CID -> oops -> family)
dayzer_family = mapping.merge(
    oops_family.rename(columns={"oops_constraint_num": "oops_constraint_num_da"}),
    on="oops_constraint_num_da", how="left").rename(columns={"oops_constraint_num_da": "oops_constraint_num"})
print('total unique ones', mapping.ConstraintId.nunique())
print("CIDs:", dayzer_family["ConstraintId"].nunique(),
      "| with family:", dayzer_family["constraint_family_num"].notna().sum(), "/", len(dayzer_family))
dayzer_family.head()

mapped_monitored = internal_dart[internal_dart['oops_constraint_num'].isin(dayzer_family.oops_constraint_num)].monitored_name.unique()
print('mapped_monitored_names are', len(mapped_monitored))
print('num of constraint family',dayzer_family.constraint_family_num.nunique())
print('num of monitored', dayzer_family.monitored_name.nunique())

total unique ones 1622
CIDs: 1622 | with family: 3374 / 3374
mapped_monitored_names are 939
num of constraint family 939
num of monitored 939


In [39]:
# russett - sbrown oops_constriaint_num is  mapped
internal_da = Constraint(market=OPEX).get_constraint_details(da_or_rt="DA")[
    ["oops_constraint_num", "monitored_clean", "contingency_clean"]].rename(
    columns={"monitored_clean": "monitored_name", "contingency_clean": "contingency_name"})
rsbrown= internal_da[internal_da['monitored_name']=='lnrussett-sbrown']
print(dayzer_family[dayzer_family['oops_constraint_num'].isin(rsbrown.oops_constraint_num)])
dayzer_family_bq = bigquery_functions.upload_to_bq_from_dataframe(
    dayzer_family,
    "temp",
    f"{OPEX}_dayzer_family{SUFFIX}",
    temp=True,
)
print("uploaded:", dayzer_family_bq)

      oops_constraint_num                       constraint_name       monitored_name                    contingency_name  ConstraintId  constraint_family_num
150                297731  SBROWN_7454_A_LN:LN RUSSETT - SBROWN  LN RUSSETT - SBROWN                                BASE       5002392                 167423
403                814690      TMP158_32185:LN RUSSETT - SBROWN  LN RUSSETT - SBROWN      CSWS:VALLIANT PITTSB9:345:1:10       3021674                 167423
1030               503828      TMP159_24149:LN RUSSETT - SBROWN  LN RUSSETT - SBROWN   OKGE:CANEY1 BODLE BROWN2:138:7:33       5001614                 167423
1318               537062      TMP159_24149:LN RUSSETT - SBROWN  LN RUSSETT - SBROWN   OKGE:BROWN2 BODLE CANEY1:138:7:33       5003688                 167423
1333               537084      TMP563_26603:LN RUSSETT - SBROWN  LN RUSSETT - SBROWN  OKGE WFEC:SUNNYSDE HUGOPP4:345:1:1       5001686                 167423
1574               603431      TMP361_28741:LN RUSSE

# Choose the map the dayzer with out oops constraint by the monitored name. Since 92% of the constraintID could be mapped with the monitored name, coverage of dayzer is there, plus monitored name could be understood as a constraint family and we control our exposure of each monitored name from this point. 

In [45]:
# --- Step C: dayzer FlowRatio per constraint_family per day (SPP) ---
# Mirrors the MISO/PJM daily block: aggregate to worst-hour FlowRatio per constraint,
# then pick the top-FlowRatio constraint (rank=1) per (dt, family).
# flow_query = f"""
# SELECT CAST(dt AS STRING) AS dt, oops_constraint_num, constraint_family_num, ShadowPrice, Flows, FlowRatio, MinFlowLimit, MaxFlowLimit
# FROM (
#   SELECT dt, ConstraintId, oops_constraint_num, constraint_family_num,
#          SUM(ShadowPrice) AS ShadowPrice, MAX(Flows) AS Flows,
#          MAX(MinFlowLimit) AS MinFlowLimit, MIN(MaxFlowLimit) AS MaxFlowLimit,
#          MAX(CASE WHEN Flows < 0 THEN Flows/NULLIF(MinFlowLimit,0) ELSE Flows/NULLIF(MaxFlowLimit,0) END) AS FlowRatio,
#          ROW_NUMBER() OVER (PARTITION BY dt, constraint_family_num ORDER BY
#              MAX(CASE WHEN Flows < 0 THEN Flows/NULLIF(MinFlowLimit,0) ELSE Flows/NULLIF(MaxFlowLimit,0) END) DESC) AS rnk
#   FROM (
#     SELECT CAST(Date AS DATE) AS dt, CAST(Hour AS INT) AS hr, a.ConstraintId,
#            b.oops_constraint_num, c.constraint_family_num,
#            ShadowPrice, Flows, MinFlowLimit, MaxFlowLimit
#     FROM `movetocloud-999.dayzer.resultConstraintVE` AS a
#     INNER JOIN `movetocloud-999.{dayzer_oops_mapping_bq}` AS b ON a.ConstraintId = b.ConstraintId
#     INNER JOIN `movetocloud-999.{oops_family_bq}`         AS c ON b.oops_constraint_num = c.oops_constraint_num
#     WHERE DATE(Date) BETWEEN DATE('{start_dt}') AND DATE('{end_dt}')
#       AND marketName = '{OPEX}' AND ScenarioId = 1
#   )
#   GROUP BY dt, ConstraintId, oops_constraint_num, constraint_family_num
# )
# ORDER BY constraint_family_num, dt
# """
# dayzer_table_bq = bigquery_functions.create_temp_table_from_query(
#     flow_query, "temp", f"{OPEX}_dayzer_table{SUFFIX}", temp=True)
# print("dayzer_table:", dayzer_table_bq)

# q = f"SELECT COUNT(*) n, COUNT(DISTINCT constraint_family_num) fams, MIN(dt) mn, MAX(dt) mx FROM `movetocloud-999.{dayzer_table_bq}`"
# print(bigquery_functions.download_df_from_bq(q).to_dict("records"))


flow_query = f"""
SELECT CAST(dt AS STRING) AS dt, constraint_family_num, monitored_name, oops_constraint_num,
       ShadowPrice, Flows, FlowRatio, MinFlowLimit, MaxFlowLimit
FROM (
  SELECT dt, constraint_family_num, monitored_name,oops_constraint_num, 
         SUM(ShadowPrice) AS ShadowPrice, MAX(Flows) AS Flows,
         MAX(MinFlowLimit) AS MinFlowLimit, MIN(MaxFlowLimit) AS MaxFlowLimit,
         MAX(CASE WHEN Flows < 0 THEN Flows/NULLIF(MinFlowLimit,0)
                  ELSE Flows/NULLIF(MaxFlowLimit,0) END) AS FlowRatio
  FROM (
    SELECT CAST(Date AS DATE) AS dt, CAST(Hour AS INT) AS hr, a.ConstraintId,
           b.constraint_family_num, b.monitored_name, b.oops_constraint_num,
           ShadowPrice, Flows, MinFlowLimit, MaxFlowLimit
    FROM `movetocloud-999.dayzer.resultConstraintVE` AS a
    INNER JOIN `movetocloud-999.{dayzer_family_bq}` AS b ON a.ConstraintId = b.ConstraintId
    WHERE DATE(Date) BETWEEN DATE('{start_dt}') AND DATE('{end_dt}')
      AND marketName = '{OPEX}' AND ScenarioId = 1
  )
  GROUP BY dt, constraint_family_num, monitored_name, oops_constraint_num
)
ORDER BY dt
"""
dayzer_table_bq = bigquery_functions.create_temp_table_from_query(
    flow_query, "temp", f"{OPEX}_dayzer_table{SUFFIX}", temp=True)


In [46]:
table = bigquery_functions.download_df_from_bq(
    f"SELECT * FROM `movetocloud-999.{dayzer_table_bq}` where dt > '2026-06-01' ORDER BY FlowRatio DESC")

# Many repetition for single constraint num per day, meaning that dayzer would give multiple predictions per day, so might need to pick the lowest mvalue for that day. Each consraintID would mapped to one oops_constraint by the monitored name. lnwr_smkhl-summ would have 17 constraintID each map to 17 oops constraint num by the monitored name. So each day has prediction of ~17 constraintID, Dayzer gives each ConstraintID 24 prediction each hour has one, we would query for each day the max flowratio of the day's prediction. We would calculate per groupby dt, constraint_fam_num, oops_constraint, max flow for that oops constraint, then we would further give the per constraint_fam level, what is max flow ratio for that day. 

In [62]:
from nighthawk.data.network.constraint import Constraint

con_df = table[['oops_constraint_num']].drop_duplicates()
d0, d1 = table['dt'].min(), table['dt'].max()

# oops_constraint_num -> monitored name
mon = pd.concat([
    Constraint(market='SPP').get_constraint_details('DA'),
    Constraint(market='SPP').get_constraint_details('RT'),
])[['oops_constraint_num', 'monitored_clean']].rename(
    columns={'monitored_clean': 'monitored_name'}
).dropna(subset=['monitored_name']).drop_duplicates('oops_constraint_num')

# DA & RT mvalue for those constraints, across the table's date range
da = Constraint(oops_constraint_num_df=con_df, market='SPP').get_mvalues(
        start_dt=d0, end_dt=d1, type='DA', granularity='daily').rename(columns={'mvalue': 'da_mvalue'})
rt = Constraint(oops_constraint_num_df=con_df, market='SPP').get_mvalues(
        start_dt=d0, end_dt=d1, type='RT', granularity='daily').rename(columns={'mvalue': 'rt_mvalue'})

dart = da[['oops_constraint_num', 'dt', 'da_mvalue']].merge(
       rt[['oops_constraint_num', 'dt', 'rt_mvalue']],
       on=['oops_constraint_num', 'dt'], how='outer')
dart['dt'] = pd.to_datetime(dart['dt']).dt.strftime('%Y-%m-%d')

# append onto the dayzer table
table_merged = (table
         .merge(mon,  on=['oops_constraint_num'], how='left')
         .merge(dart, on=['oops_constraint_num', 'dt'], how='left')).rename(columns = {'monitored_name_y':'monitored_name', 'monitored_name_x':'monitored'})

cols = ['dt', 'oops_constraint_num', 'monitored_name',
        'FlowRatio', 'Flows', 'ShadowPrice', 'da_mvalue', 'rt_mvalue']

# select the max FlowRatio for each monitored line, for example lnwr_smkhl-summ has 17 constraintID,
agg = (table_merged
       .groupby(['dt', 'monitored_name'], as_index=False)
       .agg(da_mvalue=('da_mvalue', 'sum'),
            rt_mvalue=('rt_mvalue', 'sum'),
            FlowRatio=('FlowRatio', 'max'),
            n_constraints=('oops_constraint_num', 'nunique')))


display(table_merged[cols].sort_values(['dt', 'FlowRatio'], ascending=[True, False]))
display(agg.sort_values(['dt', 'FlowRatio'], ascending=[True, False]))

,dt,oops_constraint_num,monitored_name,FlowRatio,Flows,ShadowPrice,da_mvalue,rt_mvalue
1,2026-06-02,521984.0,lnbismark2-cpc_ebis,1.550092,108.50642,-3000.0,NaN,NaN
41,2026-06-02,536899.0,lntekamho-sub1226,1.195174,161.34843,-1000.0,NaN,-269.6321
42,2026-06-02,332070.0,lntekamho-sub1226,1.195174,161.34843,-1000.0,NaN,NaN
43,2026-06-02,549700.0,lnsub1226-tekamho,1.195174,161.34843,-1000.0,NaN,NaN
44,2026-06-02,537059.0,lntekamho-sub1226,1.195174,161.34843,-1000.0,NaN,NaN
...,...,...,...,...,...,...,...,...
50365,2026-06-16,786256.0,lngracmont-anadarko,0.000000,0.00000,0.0,NaN,NaN
50366,2026-06-16,501003.0,lnexirawa-anta_tp,0.000000,0.00000,0.0,NaN,NaN
50367,2026-06-16,448553.0,xfmrcatsagr5-catsagr5,0.000000,0.00000,0.0,NaN,NaN
50368,2026-06-16,303796.0,lnstilw1-spvally2,0.000000,0.00000,0.0,NaN,NaN


,dt,monitored_name,da_mvalue,rt_mvalue,FlowRatio,n_constraints
52,2026-06-02,lnbismark2-cpc_ebis,0.0,0.0000,1.550092,1
622,2026-06-02,lnsub1226-tekamho,0.0,0.0000,1.195174,2
658,2026-06-02,lntekamho-sub1226,0.0,-269.6321,1.195174,8
141,2026-06-02,lncoonrpmu-carlntcb,0.0,0.0000,1.150790,1
154,2026-06-02,lncrmubtc1-coonrpmu,0.0,0.0000,1.150790,1
...,...,...,...,...,...,...
13983,2026-06-16,xfmrmorris-morris,0.0,0.0000,0.000000,4
13993,2026-06-16,xfmrnevada-nevada,0.0,0.0000,0.000000,1
14003,2026-06-16,xfmrpalo_dur-palo_dur,0.0,0.0000,0.000000,1
14008,2026-06-16,xfmrpioneerg-pioneerg,0.0,0.0000,0.000000,1


In [63]:
agg[(agg['dt']=='2026-06-11') & (agg['monitored_name']=='lnwr_smkhl-summ')]

,dt,monitored_name,da_mvalue,rt_mvalue,FlowRatio,n_constraints
9177,2026-06-11,lnwr_smkhl-summ,-299.6467,-916.8959,1.0,38


In [64]:
# total dayzer predicts for flowratio > 0.95
print(len(agg[((agg['da_mvalue']!=0) | (agg['rt_mvalue']!=0)) & (agg['FlowRatio'] >0.95)]))
# dayzer predicts it right when flowratio > 0.95
print(len(agg[((agg['da_mvalue']!=0) | (agg['rt_mvalue']!=0)) & (agg['FlowRatio'] >0.95) & ((agg['rt_mvalue']-agg['da_mvalue'])<=0)]))

270
96


In [60]:
agg[((agg['da_mvalue']!=0) | (agg['rt_mvalue']!=0)) & (agg['FlowRatio'] >0.95) & ((agg['rt_mvalue']-agg['da_mvalue'])<=0)]

,dt,monitored_name,da_mvalue,rt_mvalue,FlowRatio,n_constraints
122,2026-06-02,lncleoc1-cleo,0.0000,-3.1847,1.000000,2.0
296,2026-06-02,lnjarb-166thstr,0.0000,-120.3564,1.000000,2.0
468,2026-06-02,lnnwtexar-nnewbos,-51.6412,-1818.8625,0.955409,4.0
653,2026-06-02,lntahlqh5-hwy59tp,0.0000,-40.8987,1.000000,2.0
658,2026-06-02,lntekamho-sub1226,0.0000,-269.6321,1.195174,8.0
...,...,...,...,...,...,...
11991,2026-06-14,lnwr_smkhl-summ,0.0000,-3.4418,1.000000,38.0
12018,2026-06-14,xfmrcimarron-cimarron,0.0000,-66.5866,0.994977,17.0
12520,2026-06-15,lnknoll1-nhays,0.0000,-250.3222,1.000000,6.0
12719,2026-06-15,lnraun-tekamho,-1.6567,-779.3301,1.132387,8.0


In [11]:
# --- Step D: DART mvalue per family/day (SPP) + mvalue cut for families of interest ---
cf = ConstraintFamily(opexchange=OPEX, constraint_family_num_df=None)
da = cf.get_mvalues(start_dt=start_dt, end_dt=end_dt, type="DA", granularity="daily").rename(
    columns={"mvalue": "dam", "constraintFamilyNum": "constraint_family_num"})
rt = cf.get_mvalues(start_dt=start_dt, end_dt=end_dt, type="RT", granularity="daily").rename(
    columns={"mvalue": "rtm", "constraintFamilyNum": "constraint_family_num"})

dartm = pd.merge(da[["constraint_family_num", "dt", "dam"]],
                 rt[["constraint_family_num", "dt", "rtm"]],
                 on=["constraint_family_num", "dt"], how="outer")
dartm["dt"] = pd.to_datetime(dartm["dt"]).dt.strftime("%Y-%m-%d")
dartm[["dam", "rtm"]] = dartm[["dam", "rtm"]].fillna(0)

# mvalue cut: keep families that ever exceed the threshold in DA or RT
fam_max = dartm.groupby("constraint_family_num").agg(damx=("dam", "max"), rtmx=("rtm", "max")).reset_index()
interested = fam_max[(fam_max["damx"] > mvalue_threshold) | (fam_max["rtmx"] > mvalue_threshold)]["constraint_family_num"]
dartm = dartm[dartm["constraint_family_num"].isin(interested)].reset_index(drop=True)

print("families total:", fam_max.shape[0], "| above mvalue cut:", len(interested))
print("dartm rows:", len(dartm), "| date range:", dartm["dt"].min(), "->", dartm["dt"].max())
display(dartm.sort_values("dam", ascending=False).head())


families total: 2177 | above mvalue cut: 0
dartm rows: 0 | date range: nan -> nan


,constraint_family_num,dt,dam,rtm


In [ ]:
# --- Step E: INDIANAHUB ice price (proxy) + normalized mvalues ---
from nighthawk.data.pipeline.var_handler import ice_elec_price_vh
ice_df, _ = ice_elec_price_vh.get_data_and_mapping_for_ice_elec(
    [1656], "MISO", [ICE_LABEL], start_dt, end_dt, var_spec=["f"], impute=True)
ice_df = ice_df.rename(columns={f"{ICE_LABEL}_ice_elec_price_forecast_f": "ice_price"})
ice_df = ice_df.groupby("dt").agg({"ice_price": "mean"}).reset_index()
ice_df["dt"] = pd.to_datetime(ice_df["dt"]).dt.strftime("%Y-%m-%d")

dartm = dartm.merge(ice_df, on="dt", how="left")
dartm["ice_price"] = dartm["ice_price"].replace(0, np.nan).fillna(method="ffill").fillna(method="bfill")
dartm["dam_norm"] = dartm["dam"] / dartm["ice_price"]
dartm["rtm_norm"] = dartm["rtm"] / dartm["ice_price"]
print("ice_price range:", round(dartm["ice_price"].min(), 2), "->", round(dartm["ice_price"].max(), 2),
      "| nulls:", dartm["ice_price"].isna().sum())
display(dartm.head())


In [ ]:
# --- Step F: lagged rolling price-derived stats per family (avoid look-ahead) ---
# DA lagged 1 day, RT lagged 2 days (mirrors pipeline dam_minus_1 / rtm_minus_2).
dartm = dartm.sort_values(["constraint_family_num", "dt"]).reset_index(drop=True)
g = dartm.groupby("constraint_family_num")

dartm["dam_l1"]      = g["dam"].shift(1)
dartm["rtm_l2"]      = g["rtm"].shift(2)
dartm["dam_norm_l1"] = g["dam_norm"].shift(1)

def roll(col, win, fn):
    return dartm.groupby("constraint_family_num")[col].transform(
        lambda s: getattr(s.rolling(win, min_periods=1), fn)())

dartm["da_max_in7d"]      = roll("dam_l1", 7, "max")
dartm["rt_max_in7d"]      = roll("rtm_l2", 7, "max")
dartm["da_max_in30d"]     = roll("dam_l1", 30, "max")
dartm["rt_max_in30d"]     = roll("rtm_l2", 30, "max")
dartm["da_avg_in30d"]     = roll("dam_l1", 30, "mean")
dartm["da_norm_avg_in5d"] = roll("dam_norm_l1", 5, "mean")
dartm["da_rt_norm_max"]   = dartm[["dam_norm", "rtm_norm"]].max(axis=1)

print("rolling cols added. bid_dt sample:")
display(dartm[dartm["dt"] == bid_dt][
    ["constraint_family_num", "dam", "rtm", "rt_max_in7d", "da_max_in7d",
     "da_avg_in30d", "da_max_in30d", "rt_max_in30d", "da_norm_avg_in5d"]].head())


In [ ]:
# --- Step G: KV per family (rep constraint -> monitored equipment -> Powerflow KV) ---
from nighthawk.data.network.powerflow import Powerflow

rep_con = cf.get_rep_constraints().rename(columns={"constraintFamilyNum": "constraint_family_num"})
rep_con = rep_con[rep_con["constraint_family_num"].isin(interested)].copy()
rep_con["oops_constraint_num"] = rep_con["repConstraintNum"].astype(int)

c_obj = Constraint(pd.DataFrame({"oops_constraint_num": rep_con["oops_constraint_num"].unique().tolist()}), OPEX)
eq = c_obj.get_monitored_equipment_details()
eq = eq.loc[eq.groupby("oops_constraint_num")["monitored_eqNum"].idxmax()]   # one eq per constraint

pf = Powerflow(OPEX)
kv_df = pf.get_eq_name_and_kv_for_frontend(eq["monitored_eqNum"].unique().tolist())

feat = rep_con[["constraint_family_num", "oops_constraint_num"]].merge(
    eq[["oops_constraint_num", "monitored_eqNum"]].rename(columns={"monitored_eqNum": "eqNum"}),
    on="oops_constraint_num", how="left").merge(
    kv_df[["eqNum", "KV"]], on="eqNum", how="left")
feat = feat.drop_duplicates(subset="constraint_family_num")

print("families with KV:", feat["KV"].notna().sum(), "/", len(feat))
display(feat.head())


In [ ]:
# --- Step H: assemble stats (history) = price stats + FlowRatio + KV + KV_group ---
dayzer_df = bigquery_functions.download_df_from_bq(
    f"SELECT CAST(dt AS STRING) AS dt, constraint_family_num, FlowRatio FROM `movetocloud-999.{dayzer_table_bq}`")
dayzer_df["dt"] = pd.to_datetime(dayzer_df["dt"]).dt.strftime("%Y-%m-%d")

stats = dartm.merge(dayzer_df, on=["dt", "constraint_family_num"], how="left")
stats = stats.merge(feat[["constraint_family_num", "KV"]], on="constraint_family_num", how="left")

stats["FlowRatio"] = pd.to_numeric(stats["FlowRatio"], errors="coerce").fillna(0).clip(upper=1)
stats["KV"] = pd.to_numeric(stats["KV"], errors="coerce")
stats["KV_group"] = pd.cut(
    stats["KV"], bins=[-float("inf"), 1, 115, 138, 345, float("inf")],
    labels=["01_na", "02_<=115", "03_138", "04_345", "05_>=500"]).astype("object").fillna("03_138")

print("stats rows:", len(stats), "| families:", stats["constraint_family_num"].nunique())
print("KV_group counts:\n", stats.drop_duplicates("constraint_family_num")["KV_group"].value_counts())
display(stats[stats["dt"] == bid_dt][
    ["constraint_family_num", "FlowRatio", "KV", "KV_group", "da_rt_norm_max", "da_norm_avg_in5d"]].head())


In [ ]:
# --- Step I: q95 adj-norm tail loss per (family, KV_group) x FlowRatio bucket ---
def quantiles(group):
    fg = group[group["dam"] >= 0]
    def q(lo, hi):
        if hi is None:   m = fg["FlowRatio"] > lo
        elif lo is None: m = fg["FlowRatio"] <= hi
        else:            m = (fg["FlowRatio"] > lo) & (fg["FlowRatio"] <= hi)
        return fg.loc[m, "da_rt_norm_max"].sub(fg["da_norm_avg_in5d"]).clip(lower=0).quantile(0.95)
    return pd.Series({
        "adj_da_rt_norm_max_q95_0_98_1":    q(0.98, None),
        "adj_da_rt_norm_max_q95_0_95_0_98": q(0.95, 0.98),
        "adj_da_rt_norm_max_q95_0_75_0_95": q(0.75, 0.95),
        "adj_da_rt_norm_max_q95_0_0_75":    q(None, 0.75),
    })

qres = stats.groupby(["constraint_family_num", "KV_group"]).apply(quantiles).reset_index().round(2)

# monotonic enforcement: more-binding bucket >= less-binding bucket
qres["adj_da_rt_norm_max_q95_0_95_0_98"] = qres.apply(
    lambda r: r["adj_da_rt_norm_max_q95_0_75_0_95"]
    if pd.isna(r["adj_da_rt_norm_max_q95_0_95_0_98"]) or r["adj_da_rt_norm_max_q95_0_95_0_98"] < r["adj_da_rt_norm_max_q95_0_75_0_95"]
    else r["adj_da_rt_norm_max_q95_0_95_0_98"], axis=1)
qres["adj_da_rt_norm_max_q95_0_98_1"] = qres.apply(
    lambda r: r["adj_da_rt_norm_max_q95_0_95_0_98"]
    if pd.isna(r["adj_da_rt_norm_max_q95_0_98_1"]) or r["adj_da_rt_norm_max_q95_0_98_1"] < r["adj_da_rt_norm_max_q95_0_95_0_98"]
    else r["adj_da_rt_norm_max_q95_0_98_1"], axis=1)

print("quantile rows (family x KV_group):", len(qres))
display(qres.head())


In [ ]:
# --- Step J: TARGET TABLE #2 — SPP ..._with_quantiles (bid_dt) ---
qcols = ["adj_da_rt_norm_max_q95_0_98_1", "adj_da_rt_norm_max_q95_0_95_0_98",
         "adj_da_rt_norm_max_q95_0_75_0_95", "adj_da_rt_norm_max_q95_0_0_75"]

with_quantiles = (stats[stats["dt"] == bid_dt][["dt", "constraint_family_num", "KV_group"]]
                  .merge(qres, on=["constraint_family_num", "KV_group"], how="left")[["dt", "constraint_family_num"] + qcols])

with_quantiles_bq = bigquery_functions.upload_to_bq_from_dataframe(
    with_quantiles, "temp", f"{OPEX}_constraint_family_exposure_with_quantiles{SUFFIX}", temp=True)
print("TARGET #2 with_quantiles:", with_quantiles_bq, "| rows:", len(with_quantiles))
display(with_quantiles.head())


In [ ]:
# --- Step K: SPP portfolio + node-level dfax (per family) ---
# NOTE: get_dfax_on_all_nodes over the interested rep-constraints is the heaviest step.
from nighthawk.data.product import ve as ve_mod
from nighthawk.data.product.ve import DailyBidsManager

portfolio = DailyBidsManager(opexchange=OPEX, bid_date=bid_dt).get_bids_from_table(label="preautomated_cuts")
print("portfolio rows:", len(portfolio), "| strategies:", portfolio["strategy"].unique() if len(portfolio) else "-")

# node-level dfax for the interested families' representative constraints
c_obj_dfax = Constraint(pd.DataFrame({"oops_constraint_num": rep_con["oops_constraint_num"].unique().tolist()}), OPEX)
c_dfax = c_obj_dfax.get_dfax_on_all_nodes(dfax_cutoff=0.01)
ve_obj = ve_mod.VE(OPEX)
biddable = ve_obj.get_biddable_nodes_for_daterange(bid_dt, bid_dt)
c_dfax = c_dfax[c_dfax["node_num"].isin(biddable["node_num"].unique())]

dfax = pd.merge(rep_con[["constraint_family_num", "oops_constraint_num"]], c_dfax, on="oops_constraint_num", how="inner")
# representative (max |dfax|) per (family, node)
idx = dfax.groupby(["constraint_family_num", "node_num"])["dfax"].apply(lambda x: x.abs().idxmax())
dfax = dfax.loc[idx, ["constraint_family_num", "oops_constraint_num", "node_num", "dfax"]].reset_index(drop=True)
print("dfax rows:", len(dfax), "| families:", dfax["constraint_family_num"].nunique(),
      "| nodes:", dfax["node_num"].nunique())
display(dfax.head())


In [ ]:
# --- Step L: TARGET TABLE #1 — SPP constraint_family_exposure (bid_dt) ---
# dfax-weighted short/long exposure per family (mirrors get_portfolio_exposure_for_future_bids).
portfolio["inc_mw"] = np.where(portfolio["incdec"] == "Increment", portfolio["bid_mw"], 0.0)
portfolio["dec_mw"] = np.where(portfolio["incdec"] == "Decrement", portfolio["bid_mw"], 0.0)
bs = portfolio.groupby(["dt", "node_num"], as_index=False).agg(
    bid_mw_inc=("inc_mw", "sum"), bid_mw_dec=("dec_mw", "sum"))
bs["dt"] = pd.to_datetime(bs["dt"]).dt.strftime("%Y-%m-%d")

ex = bs.merge(dfax[["constraint_family_num", "node_num", "dfax"]], on="node_num", how="inner")
ex["short"] = np.where(ex["dfax"] < 0, -ex["dfax"] * ex["bid_mw_dec"],  ex["dfax"] * ex["bid_mw_inc"])
ex["long"]  = np.where(ex["dfax"] < 0, -ex["dfax"] * ex["bid_mw_inc"],  ex["dfax"] * ex["bid_mw_dec"])
short_bid = ex.groupby(["dt", "constraint_family_num"], as_index=False).agg(
    short_bid_mw=("short", "sum"), long_bid_mw=("long", "sum"))

keep = ["dt", "constraint_family_num", "oops_constraint_num", "KV", "KV_group", "FlowRatio",
        "rt_max_in7d", "da_max_in7d", "da_avg_in30d", "da_max_in30d", "rt_max_in30d",
        "dam", "rtm", "dam_norm", "rtm_norm"]
exposure = stats[stats["dt"] == bid_dt].merge(feat[["constraint_family_num", "oops_constraint_num"]],
                                              on="constraint_family_num", how="left")
exposure = exposure[[c for c in keep if c in exposure.columns]].merge(
    short_bid, on=["dt", "constraint_family_num"], how="left")
exposure[["short_bid_mw", "long_bid_mw"]] = exposure[["short_bid_mw", "long_bid_mw"]].fillna(0)

exposure_bq = bigquery_functions.upload_to_bq_from_dataframe(
    exposure, "temp", f"{OPEX}_constraint_family_exposure{SUFFIX}", temp=True)
print("TARGET #1 exposure:", exposure_bq, "| rows:", len(exposure),
      "| families with short>0:", (exposure["short_bid_mw"] > 0).sum())
display(exposure.sort_values("short_bid_mw", ascending=False).head())


In [ ]:
# --- Step M: risk-cut math -> final_factor per family (RC-6 .. RC-9) ---
all_dt = exposure.merge(with_quantiles, on=["dt", "constraint_family_num"], how="inner")
all_dt = all_dt[((all_dt["da_max_in30d"] > 100) | (all_dt["rt_max_in30d"] > 100)) & (all_dt["short_bid_mw"] > 0)].copy()
all_dt = all_dt.merge(ice_df, on="dt", how="left")
all_dt["FlowRatio"] = pd.to_numeric(all_dt["FlowRatio"], errors="coerce").fillna(0).clip(upper=1)

scale = 3
base_limit = 1_000_000 * scale
all_dt["risk_limit"] = base_limit / all_dt["ice_price"]

kv = pd.to_numeric(all_dt["KV"], errors="coerce"); kv_clean = kv.mask(kv <= 0)
all_dt["kv_factor"] = np.select(
    [kv_clean.le(69), kv_clean.le(115), kv_clean.le(161), kv_clean.le(220), kv_clean.le(345), kv_clean.le(500), kv_clean.gt(500)],
    [0.7, 1.1, 1.6, 2.1, 2.6, 3.1, 3.6], default=2.0)
all_dt["risk_limit"] = all_dt["risk_limit"] * all_dt["kv_factor"]

FR, KG = all_dt["FlowRatio"], all_dt["KV_group"]
def band(lo, hi):
    if hi is None: return FR > lo
    if lo is None: return FR <= hi
    return (FR > lo) & (FR <= hi)
floors = {"02_<=115": (46, 26, 20, 1), "03_138": (42, 28, 20, 3), "04_345": (35, 21, 17, 2), "05_>=500": (43, 28, 24, 1)}
rules = []
for kg, (f98, f95, f75, f0) in floors.items():
    rules += [
        (band(0.98, None) & (KG == kg), all_dt["adj_da_rt_norm_max_q95_0_98_1"].apply(lambda x, f=f98: f if pd.isna(x) or x < f * 0.2 else x)),
        (band(0.95, 0.98) & (KG == kg), all_dt["adj_da_rt_norm_max_q95_0_95_0_98"].fillna(f95)),
        (band(0.75, 0.95) & (KG == kg), all_dt["adj_da_rt_norm_max_q95_0_75_0_95"].fillna(f75)),
        (band(None, 0.75) & (KG == kg), all_dt["adj_da_rt_norm_max_q95_0_0_75"].fillna(f0)),
    ]
all_dt["risk_per_mw"] = np.select([c for c, _ in rules], [v for _, v in rules], default=0.1)
all_dt["risk_per_mw_recent"] = np.where(
    (all_dt["FlowRatio"] > 0.98) | (all_dt["FlowRatio"].isna() & (all_dt["rt_max_in7d"] > 10 * all_dt["da_avg_in30d"]) & (all_dt["rt_max_in7d"] > 400)),
    np.maximum(all_dt["rt_max_in7d"] / all_dt["ice_price"] - all_dt["da_max_in7d"] / all_dt["ice_price"], 0.1), 0.1)

all_dt["long_term_mw_limit"] = all_dt["risk_limit"] / all_dt["risk_per_mw"]
all_dt["recent_mw_limit"]    = all_dt["risk_limit"] / all_dt["risk_per_mw_recent"]
all_dt["KV_mw_limit"]        = 1000 * scale * all_dt["kv_factor"]

nz = all_dt["short_bid_mw"] != 0
for col, lim in [("long_term_risk_scale_factor", "long_term_mw_limit"),
                 ("recent_risk_scale_factor", "recent_mw_limit"),
                 ("kv_mw_scale_factor", "KV_mw_limit")]:
    all_dt[col] = 1.0
    all_dt.loc[nz, col] = np.minimum(all_dt.loc[nz, "short_bid_mw"], all_dt.loc[nz, lim]) / all_dt.loc[nz, "short_bid_mw"]
all_dt["final_factor"] = all_dt[["long_term_risk_scale_factor", "recent_risk_scale_factor", "kv_mw_scale_factor"]].min(axis=1)

print("families considered:", len(all_dt), "| being cut (final_factor<1):", (all_dt["final_factor"] < 1).sum())
display(all_dt[["constraint_family_num", "FlowRatio", "KV_group", "short_bid_mw",
                "risk_per_mw", "long_term_mw_limit", "final_factor"]].sort_values("final_factor").head(15))


In [ ]:
# --- Step N: apply final_factor to portfolio via dfax (RC-11) + summary ---
cut_kv = ["02_<=115", "03_138", "04_345", "05_>=500"]
ff = all_dt[["constraint_family_num", "final_factor"]]
d = dfax.merge(ff, on="constraint_family_num", how="left")
d["final_factor"] = d["final_factor"].fillna(1.0)
# KV_group per family (from exposure); families not cut -> factor stays 1 regardless
d = d.merge(exposure[["constraint_family_num", "KV_group"]].drop_duplicates(), on="constraint_family_num", how="left")
d = d[d["dfax"].abs() > 0.05]
d["inc_factor"] = np.where((d["dfax"] > 0) & (d["KV_group"].isin(cut_kv)), d["final_factor"], 1.0)
d["dec_factor"] = np.where((d["dfax"] < 0) & (d["KV_group"].isin(cut_kv)), d["final_factor"], 1.0)
node_factor = d.groupby("node_num", as_index=False).agg(inc_factor=("inc_factor", "min"), dec_factor=("dec_factor", "min"))

ps = portfolio.merge(node_factor, on="node_num", how="left")
ps["inc_factor"] = ps["inc_factor"].fillna(1.0)
ps["dec_factor"] = ps["dec_factor"].fillna(1.0)
ps["bid_mw_original"] = ps["bid_mw"]
ps["bid_mw"] = np.where(ps["incdec"] == "Increment", ps["bid_mw"] * ps["inc_factor"], ps["bid_mw"] * ps["dec_factor"])

diff = ps.groupby("incdec").agg(orig=("bid_mw_original", "sum"), scaled=("bid_mw", "sum")).reset_index()
diff["reduction_pct"] = (1 - diff["scaled"] / diff["orig"]) * 100
print("=== SPP risk cut — reduction by incdec ===")
display(diff)
print("\nTop affected nodes:")
nd = ps.groupby(["node_num", "incdec"]).agg(orig=("bid_mw_original", "sum"), scaled=("bid_mw", "sum")).reset_index()
nd["reduction"] = nd["orig"] - nd["scaled"]
display(nd.sort_values("reduction", ascending=False).head(15))
